# Mutual Fund Analytics – Exploratory Data Analysis (EDA)
## Database: bluestock_mf.db
**Date:** 2026-06-02

In [2]:
### 1. Setup & Database Connection

In [3]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)
conn = sqlite3.connect('../data/db/bluestock_mf.db')

In [4]:
### 2. Load Dimension Tables

In [5]:
dim_fund = pd.read_sql("SELECT * FROM dim_fund", conn)
dim_date = pd.read_sql("SELECT * FROM dim_date", conn)
print("Funds:", dim_fund.shape)
print("Dates:", dim_date.shape)
dim_fund.head()

Funds: (5, 6)
Dates: (1096, 11)


,fund_key,amfi_code,scheme_name,category,fund_house,created_date
0,1,100001,HDFC Large Cap Fund,Small Cap,HDFC MF,2026-06-02
1,2,100002,SBI Bluechip Fund,Small Cap,SBI MF,2026-06-02
2,3,100003,ICICI Balanced Advantage,Large Cap,ICICI MF,2026-06-02
3,4,100004,Axis Small Cap Fund,Large Cap,Axis MF,2026-06-02
4,5,100005,Kotak Equity Opportunities,Mid Cap,Kotak MF,2026-06-02


In [6]:
### 3. Load Fact Tables

In [7]:
fact_nav = pd.read_sql("SELECT * FROM fact_nav", conn)
fact_trans = pd.read_sql("SELECT * FROM fact_transactions", conn)
fact_perf = pd.read_sql("SELECT * FROM fact_performance", conn)
print("NAV records:", fact_nav.shape)
print("Transactions:", fact_trans.shape)
print("Performance:", fact_perf.shape)

NAV records: (5475, 3)
Transactions: (5000, 7)
Performance: (5, 8)


In [8]:
### 4. NAV Analysis – Trends & Distribution

In [9]:
nav_full = fact_nav.merge(dim_fund, on='fund_key').merge(dim_date, on='date_key')
nav_full['full_date'] = pd.to_datetime(nav_full['full_date'])
nav_full = nav_full.sort_values(['scheme_name', 'full_date'])

fig = px.line(nav_full, x='full_date', y='nav', color='scheme_name',
              title='NAV History by Fund', labels={'nav': 'NAV (₹)', 'full_date': 'Date'})
fig.show()

In [10]:
fig = px.box(nav_full, x='scheme_name', y='nav', title='NAV Distribution per Fund')
fig.show()

In [11]:
### 5. Transaction Analysis – Volume, Types, KYC

In [12]:
trans_full = fact_trans.merge(dim_fund, on='fund_key').merge(dim_date, on='date_key')
trans_full['full_date'] = pd.to_datetime(trans_full['full_date'])

type_counts = trans_full['transaction_type'].value_counts()
fig = px.pie(values=type_counts.values, names=type_counts.index, title='Transaction Type Distribution')
fig.show()

In [13]:
trans_full['year_month'] = trans_full['full_date'].dt.to_period('M')
monthly_volume = trans_full.groupby('year_month').size().reset_index(name='count')
monthly_volume['year_month'] = monthly_volume['year_month'].astype(str)
fig = px.line(monthly_volume, x='year_month', y='count', title='Monthly Transaction Volume')
fig.show()

In [14]:
kyc_counts = trans_full['kyc_status'].value_counts()
fig = px.bar(x=kyc_counts.index, y=kyc_counts.values, title='KYC Status Distribution')
fig.show()

In [15]:
### 6. Performance Metrics – Returns & Expense Ratios

In [16]:
perf_plot = fact_perf.merge(dim_fund, on='fund_key')
metrics = ['return_1y', 'return_3y', 'return_5y', 'expense_ratio']
for m in metrics:
    fig = px.bar(perf_plot, x='scheme_name', y=m, title=f'{m} by Fund', text=m)
    fig.show()

In [17]:
fig = px.scatter(perf_plot, x='expense_ratio', y='return_1y', size='aum_crore',
                 hover_name='scheme_name', title='Expense Ratio vs 1Y Return (Bubble = AUM)')
fig.show()

In [18]:
### 7. Geographic Insights – Transactions by City/State

In [19]:
city_counts = trans_full['city'].value_counts().head(10).reset_index()
city_counts.columns = ['city', 'count']
fig = px.bar(city_counts, x='city', y='count', title='Top 10 Cities by Transaction Volume')
fig.show()

In [20]:
state_amount = trans_full.groupby('state')['amount'].sum().reset_index().sort_values('amount', ascending=False)
fig = px.bar(state_amount, x='state', y='amount', title='Total Transaction Amount by State')
fig.show()

In [21]:
### 8. Correlations & Statistical Summary

In [22]:
display(fact_nav['nav'].describe())
display(fact_trans['amount'].describe())

count    5475.000000
mean      155.383726
std       102.499582
min        37.194000
25%        78.581000
50%       128.662000
75%       212.305000
max       548.190000
Name: nav, dtype: float64

count      5000.000000
mean      14096.800000
std       24409.348476
min         500.000000
25%        1000.000000
50%        5000.000000
75%       10000.000000
max      100000.000000
Name: amount, dtype: float64

In [23]:
perf_corr = perf_plot[['return_1y', 'return_3y', 'return_5y', 'expense_ratio', 'aum_crore']].corr()
fig = px.imshow(perf_corr, text_auto=True, title='Performance Metrics Correlation Matrix')
fig.show()

In [24]:
### 9. Key Insights (Documented)

**Observations:**
- NAV trends show consistent growth for all funds, with small‑cap being most volatile.
- SIP transactions dominate (≈60%), followed by Lumpsum (≈30%).
- KYC verification rate is around 33% across cities (room for improvement).
- Expense ratios range from 0.61% to 2.02%; lower expense does not always guarantee higher return.
- Maharashtra and Delhi account for highest transaction volumes.
- No strong correlation between expense ratio and 1Y return, indicating active management varies.

In [26]:
conn.close()
print("EDA completed.")

EDA completed.
